# 📝 Data Contract: FlyRank Internship Warehouse

## 1. Dataset Overview
- **Source:** `FlyRank/internship-warehouse` (Release v20260703)
- **Scale:** ~81.8 million total rows
- **Format:** Star schema with salted, namespaced, fingerprinted hash keys

## 2. Core Entities & Schema
We will be querying the following tables:
- `dim_clients`: One row per pseudonymized client (104 clients).
- `dim_content`: One row per pseudonymized content item (519,606 items).
- `fact_content_daily_performance`: One row per report date, client, and content item (~78.8M rows, partitioned by month).
- `fact_content_query_90d`: Query-level data with salted hashes over a fixed 90-day window (~2.4M rows).

## 3. Data Security & Usage Terms
By accessing this dataset, I agree to the FlyRank Internship Data Use Terms:
- **Permitted Use:** Anonymized research and education use only.
- **Zero Re-identification:** No attempts to re-identify clients, domains, queries, keywords, or content.
- **No Redistribution:** Raw data files will not be redistributed or committed to version control.
- **Clean Outputs:** No client-identifying data will appear in any public output, including case studies, charts, repositories, or demos.

## 1. Unit of analysis + time window

In `fact_content_daily_performance`, one row equals one report date, for one pseudonymized client, for one pseudonymized content item. The dataset is an unbalanced panel built from a frozen snapshot on 2026-07-03, meaning the history depth varies per client based on their onboarding date.

In [5]:
import os
import duckdb
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

# Verify grain (Expect 0 rows if grain is truly unique per date/client/content)
query_1 = f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as row_count
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5;
"""
print("Checking for duplicate grains (Expect empty table):")
print(con.sql(query_1))

Checking for duplicate grains (Expect empty table):
┌─────────────┬────────────────┬─────────────────┬───────────┐
│ report_date │ client_hash_id │ content_hash_id │ row_count │
│    date     │    varchar     │     varchar     │   int64   │
└─────────────┴────────────────┴─────────────────┴───────────┘
                            0 rows                          



## 2. Fields: feature / label / context / excluded

- **Features:** `client_hash_id`, `content_hash`, `device_type`, `region`
- **Label:** `performance_score` (daily target metric)
- **Context:** `report_date`
- **Excluded:** `internal_id` (raw PII, not needed for research), `raw_query_string` (too high-cardinality/PII)

In [6]:
query_schema = f"""
DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 1;
"""
print("Fields in fact_content_daily_performance:")
print(con.sql(query_schema))

Fields in fact_content_daily_performance:
┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT     

## 3. Verify it with queries (grain, counts, missing values, windows)

We verified total row counts, distinct clients, and checked for missing values in key metrics.

In [7]:
# Verify total counts and check for missing values in key metrics
query_verify = f"""
SELECT 
    COUNT(*) as total_rows,
    COUNT(DISTINCT client_hash_id) as unique_clients,
    SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) as missing_impressions
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet');
"""
print("Verification of row counts, distinct clients, and missing data:")
print(con.sql(query_verify))

Verification of row counts, distinct clients, and missing data:
┌────────────┬────────────────┬─────────────────────┐
│ total_rows │ unique_clients │ missing_impressions │
│   int64    │     int64      │       int128        │
├────────────┼────────────────┼─────────────────────┤
│    9841378 │             55 │                   0 │
└────────────┴────────────────┴─────────────────────┘



## 4. Data limits

The data exhibits unbalanced history; client activity varies significantly over time based on their onboarding.

In [8]:
# Prove the unbalanced history limit by checking active clients per month
query_limits = f"""
SELECT 
    date_trunc('month', CAST(report_date AS DATE)) as report_month,
    COUNT(DISTINCT client_hash_id) as active_clients
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
GROUP BY 1
ORDER BY 1 DESC
LIMIT 12;
"""
print("Active clients per month (Proving unbalanced history):")
print(con.sql(query_limits))

Active clients per month (Proving unbalanced history):
┌─────────────────────┬────────────────┐
│    report_month     │ active_clients │
│      timestamp      │     int64      │
├─────────────────────┼────────────────┤
│ 2026-06-01 00:00:00 │             65 │
│ 2026-05-01 00:00:00 │             66 │
│ 2026-04-01 00:00:00 │             61 │
│ 2026-03-01 00:00:00 │             55 │
│ 2026-02-01 00:00:00 │             54 │
│ 2026-01-01 00:00:00 │             42 │
│ 2025-12-01 00:00:00 │             43 │
│ 2025-11-01 00:00:00 │             43 │
│ 2025-10-01 00:00:00 │             31 │
│ 2025-09-01 00:00:00 │             23 │
│ 2025-08-01 00:00:00 │             15 │
│ 2025-07-01 00:00:00 │             16 │
└─────────────────────┴────────────────┘
  12 rows                    2 columns



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.